In [ ]:
# ─────────────────────────────────────────────
# PART 1: Install & Imports
# ─────────────────────────────────────────────
!pip install open3d plotly huggingface_hub scikit-image numpy scipy wandb -q
!pip install torch scikit-learn -q

import open3d as o3d
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import wandb
import os
import zipfile
import math
import time
from sklearn.decomposition import PCA
from huggingface_hub import hf_hub_download
from torch.utils.data import Dataset, DataLoader, Subset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.9 MB/s eta 0:00:00
Using device: cuda


In [ ]:
# ─────────────────────────────────────────────
# PART 2: Download & Extract Dataset
# ─────────────────────────────────────────────
zip_path = hf_hub_download(
    repo_id="BGLab/AgriField3D",
    filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
    repo_type="dataset",
    local_dir="./data"
)

extract_dir = "./data/RawPCD_10k"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

ply_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(extract_dir)
    for f in files if f.endswith('.ply')
])
print(f"Total plants found: {len(ply_files)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

Total plants found: 1045


In [ ]:
# ─────────────────────────────────────────────
# PART 3: Dataset + Fixed 500/Rest Split
# ─────────────────────────────────────────────
NUM_POINTS = 1024
TRAIN_SIZE = 500

def preprocess(ply_path, num_points=NUM_POINTS):
    pcd = o3d.io.read_point_cloud(ply_path)
    pts = np.asarray(pcd.points)
    if pts.shape[0] == 0:
        return None
    N   = pts.shape[0]
    idx = (np.random.choice(N, num_points, replace=False)
           if N >= num_points
           else np.random.choice(N, num_points, replace=True))
    pts = pts[idx]
    pts -= pts.mean(axis=0)
    pts /= (np.linalg.norm(pts, axis=1).max() + 1e-8)
    return pts.astype(np.float32)


class MaizeDataset(Dataset):
    def __init__(self, ply_files):
        self.samples = []
        self.paths   = []
        print(f"Loading {len(ply_files)} files ...")
        for i, path in enumerate(ply_files):
            try:
                pts = preprocess(path)
                if pts is not None:
                    self.samples.append(torch.tensor(pts))
                    self.paths.append(path)
            except Exception as e:
                print(f"  [skip] {os.path.basename(path)}: {e}")
            if (i+1) % 20 == 0:
                print(f"  {i+1}/{len(ply_files)} -- {len(self.samples)} valid")
        print(f"Dataset ready: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], idx


# Build & split
full_dataset  = MaizeDataset(ply_files)
total         = len(full_dataset)

all_indices   = np.random.permutation(total)
train_indices = all_indices[:TRAIN_SIZE].tolist()
test_indices  = all_indices[TRAIN_SIZE:].tolist()

train_dataset = Subset(full_dataset, train_indices)
test_dataset  = Subset(full_dataset, test_indices)

# num_workers=2 + pin_memory speeds up GPU transfer
train_loader  = DataLoader(train_dataset, batch_size=8,
                           shuffle=True,  drop_last=True,
                           num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=8,
                           shuffle=False, drop_last=False,
                           num_workers=2, pin_memory=True)

print(f"Train : {len(train_dataset)}")
print(f"Test  : {len(test_dataset)} (unseen)")


Loading 1045 files ...
  20/1045 -- 20 valid
  40/1045 -- 40 valid
  60/1045 -- 60 valid
  80/1045 -- 80 valid
  100/1045 -- 100 valid
  120/1045 -- 120 valid
  140/1045 -- 140 valid
  160/1045 -- 160 valid
  180/1045 -- 180 valid
  200/1045 -- 200 valid
  220/1045 -- 220 valid
  240/1045 -- 240 valid
  260/1045 -- 260 valid
  280/1045 -- 280 valid
  300/1045 -- 300 valid
  320/1045 -- 320 valid
  340/1045 -- 340 valid
  360/1045 -- 360 valid
  380/1045 -- 380 valid
  400/1045 -- 400 valid
  420/1045 -- 420 valid
  440/1045 -- 440 valid
  460/1045 -- 460 valid
  480/1045 -- 480 valid
  500/1045 -- 500 valid
  520/1045 -- 520 valid
  540/1045 -- 540 valid
  560/1045 -- 560 valid
  580/1045 -- 580 valid
  600/1045 -- 600 valid
  620/1045 -- 620 valid
  640/1045 -- 640 valid
  660/1045 -- 660 valid
  680/1045 -- 680 valid
  700/1045 -- 700 valid
  720/1045 -- 720 valid
  740/1045 -- 740 valid
  760/1045 -- 760 valid
  780/1045 -- 780 valid
  800/1045 -- 800 valid
  820/1045 -- 820 valid
 

In [ ]:
# ─────────────────────────────────────────────
# PART 4: PointNet++ Model — Pure PyTorch
# Optimisations vs original:
#   * Latent dim 256 -> 512 (richer representation)
#   * AdamW with weight decay in training cell
#   * Dropout in encoder + decoder (better generalisation)
#   * Wider decoder MLP (1024->2048 hidden units)
#   * LR warm-up + cosine annealing for stable convergence
# ─────────────────────────────────────────────

def farthest_point_sampling(xyz, n_samples):
    B, N, _ = xyz.shape
    selected    = torch.zeros(B, n_samples, dtype=torch.long, device=xyz.device)
    selected[:, 0] = torch.randint(0, N, (B,), device=xyz.device)
    dist = torch.full((B, N), float('inf'), device=xyz.device)
    for i in range(1, n_samples):
        prev     = selected[:, i-1]
        prev_xyz = xyz[torch.arange(B), prev, :]
        d        = ((xyz - prev_xyz.unsqueeze(1)).pow(2).sum(-1))
        dist     = torch.min(dist, d)
        selected[:, i] = dist.argmax(dim=1)
    return selected


def gather_points(xyz, idx):
    B, N, C = xyz.shape
    M       = idx.shape[1]
    idx_exp = idx.unsqueeze(-1).expand(B, M, C)
    return xyz.gather(1, idx_exp)


def ball_query(xyz, query_xyz, radius, max_samples):
    B, N, _ = xyz.shape
    M       = query_xyz.shape[1]
    diff  = query_xyz.unsqueeze(2) - xyz.unsqueeze(1)
    dist  = diff.pow(2).sum(-1)
    idx   = dist.argsort(dim=-1)[:, :, :max_samples]
    nearest       = idx[:, :, 0:1].expand_as(idx)
    within_radius = dist.gather(2, idx) < radius ** 2
    idx           = torch.where(within_radius, idx, nearest)
    return idx


class SetAbstraction(nn.Module):
    def __init__(self, n_centroids, radius, max_samples, in_channels, mlp_channels):
        super().__init__()
        self.n_centroids = n_centroids
        self.radius      = radius
        self.max_samples = max_samples
        layers = []
        in_ch  = in_channels + 3
        for out_ch in mlp_channels:
            layers += [nn.Conv2d(in_ch, out_ch, 1),
                       nn.BatchNorm2d(out_ch), nn.ReLU()]
            in_ch = out_ch
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, features=None):
        B, N, _ = xyz.shape
        M = self.n_centroids
        fps_idx   = farthest_point_sampling(xyz, M)
        new_xyz   = gather_points(xyz, fps_idx)
        group_idx = ball_query(xyz, new_xyz, self.radius, self.max_samples)
        K = group_idx.shape[2]
        grouped_xyz = gather_points(xyz, group_idx.view(B, -1)).view(B, M, K, 3)
        grouped_xyz -= new_xyz.unsqueeze(2)
        if features is not None:
            grouped_feat = gather_points(features, group_idx.view(B, -1)).view(B, M, K, -1)
            grouped = torch.cat([grouped_xyz, grouped_feat], dim=-1)
        else:
            grouped = grouped_xyz
        grouped  = grouped.permute(0, 3, 1, 2)
        grouped  = self.mlp(grouped)
        new_feat = grouped.max(dim=-1)[0].permute(0, 2, 1)
        return new_xyz, new_feat


class FeaturePropagation(nn.Module):
    def __init__(self, in_channels, mlp_channels):
        super().__init__()
        layers = []
        in_ch  = in_channels
        for out_ch in mlp_channels:
            layers += [nn.Conv1d(in_ch, out_ch, 1),
                       nn.BatchNorm1d(out_ch), nn.ReLU()]
            in_ch = out_ch
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz1, xyz2, feat1, feat2):
        B, N, _ = xyz1.shape
        M = xyz2.shape[1]
        diff  = xyz1.unsqueeze(2) - xyz2.unsqueeze(1)
        dist  = diff.pow(2).sum(-1).clamp(min=1e-10).sqrt()
        k     = min(3, M)
        knn_d, knn_i = dist.topk(k, dim=-1, largest=False)
        weight = 1.0 / knn_d.clamp(min=1e-8)
        weight = weight / weight.sum(dim=-1, keepdim=True)
        knn_i_exp = knn_i.unsqueeze(-1).expand(B, N, k, feat2.shape[-1])
        interp    = (feat2.unsqueeze(1).expand(B, N, M, -1).gather(2, knn_i_exp))
        interp    = (interp * weight.unsqueeze(-1)).sum(dim=2)
        if feat1 is not None:
            new_feat = torch.cat([feat1, interp], dim=-1)
        else:
            new_feat = interp
        new_feat = new_feat.permute(0, 2, 1)
        new_feat = self.mlp(new_feat)
        return new_feat.permute(0, 2, 1)


class PointNetPP_Autoencoder(nn.Module):
    def __init__(self, num_points=NUM_POINTS, latent_dim=512):
        super().__init__()
        self.num_points = num_points

        # Encoder: 3 hierarchical Set Abstraction layers
        self.sa1 = SetAbstraction(512, 0.2, 32,  0,   [64,  64,  128])
        self.sa2 = SetAbstraction(128, 0.4, 64,  128, [128, 128, 256])
        self.sa3 = SetAbstraction(32,  0.8, 128, 256, [256, 256, 512])

        # Global pooling -> latent (with dropout for regularisation)
        self.global_fc = nn.Sequential(
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, latent_dim)
        )

        # Decoder: wider MLP with dropout
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 2048), nn.BatchNorm1d(2048), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(2048, num_points * 3)
        )

    def encode(self, xyz):
        xyz1, f1 = self.sa1(xyz)
        xyz2, f2 = self.sa2(xyz1, f1)
        xyz3, f3 = self.sa3(xyz2, f2)
        g = f3.max(dim=1)[0]
        return self.global_fc(g)

    def decode(self, z):
        return self.decoder(z).view(-1, self.num_points, 3)

    def forward(self, xyz):
        z     = self.encode(xyz)
        recon = self.decode(z)
        return recon, z


def chamfer_distance(pred, target):
    d1 = (pred.unsqueeze(2) - target.unsqueeze(1)).pow(2).sum(-1)
    d2 = (target.unsqueeze(2) - pred.unsqueeze(1)).pow(2).sum(-1)
    return d1.min(2)[0].mean() + d2.min(2)[0].mean()


In [ ]:
# ─────────────────────────────────────────────
# PART 5: Checkpoint Utilities
# ─────────────────────────────────────────────
CKPT_DIR = "./checkpoints_pnetpp"
os.makedirs(CKPT_DIR, exist_ok=True)

def save_checkpoint(model, optimizer, scheduler, epoch, loss, tag="latest"):
    path = os.path.join(CKPT_DIR, f"pnetpp_{tag}.pt")
    torch.save({
        "epoch"       : epoch,
        "model_state" : model.state_dict(),
        "optim_state" : optimizer.state_dict(),
        "sched_state" : scheduler.state_dict(),
        "loss"        : loss,
    }, path)
    return path


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if optimizer: optimizer.load_state_dict(ckpt["optim_state"])
    if scheduler: scheduler.load_state_dict(ckpt["sched_state"])
    print(f"  Loaded -- epoch {ckpt['epoch']} | loss {ckpt['loss']:.6f}")
    return ckpt


In [ ]:
# ─────────────────────────────────────────────
# PART 6: Training (300 epochs, <= 1 hr)
# Key changes:
#   * 500 -> 300 epochs  (still converges; faster)
#   * AdamW + weight_decay=1e-4 (better regularisation)
#   * LR warm-up (10 ep) + cosine decay
#   * Log-scale loss logged to W&B every step
#   * Dual linear+log loss curve sent to W&B
# ─────────────────────────────────────────────
CONFIG = dict(
    epochs           = 300,
    batch_size       = 8,
    lr               = 1e-3,
    weight_decay     = 1e-4,
    num_points       = NUM_POINTS,
    latent_dim       = 512,
    train_size       = TRAIN_SIZE,
    checkpoint_every = 50,
    warmup_epochs    = 10,
)

wandb.init(
    project = "maize-pointnetpp",
    config  = CONFIG,
    name    = "pnetpp-optimised-300ep"
)

model = PointNetPP_Autoencoder(
    num_points=NUM_POINTS,
    latent_dim=CONFIG["latent_dim"]
).to(device)

# AdamW for better weight regularisation vs plain Adam
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"]
)

# Linear warm-up then cosine decay
def lr_lambda(epoch):
    if epoch < CONFIG["warmup_epochs"]:
        return epoch / max(1, CONFIG["warmup_epochs"])
    progress = (epoch - CONFIG["warmup_epochs"]) / max(
        1, CONFIG["epochs"] - CONFIG["warmup_epochs"])
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

wandb.watch(model, log="all", log_freq=20)

best_loss    = float("inf")
train_losses = []
training_start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):

    # ── Train ──────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for batch, _ in train_loader:
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad()
        recon, _ = model(batch)
        loss      = chamfer_distance(recon, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)
    scheduler.step()

    cur_lr  = optimizer.param_groups[0]["lr"]
    elapsed = (time.time() - training_start) / 60   # minutes

    # ── W&B: log both linear loss AND log10 loss every epoch ───────
    wandb.log({
        "epoch"          : epoch,
        "train_loss"     : epoch_loss,                      # linear
        "train_loss_log" : np.log10(epoch_loss + 1e-10),   # log10
        "lr"             : cur_lr,
        "elapsed_min"    : elapsed,
    })

    # ── Periodic checkpoint ────────────────────────────────────────
    if epoch % CONFIG["checkpoint_every"] == 0:
        path = save_checkpoint(model, optimizer, scheduler,
                               epoch, epoch_loss, tag="latest")
        wandb.save(path)
        print(f"Epoch {epoch:4d} | Loss {epoch_loss:.6f} | "
              f"LR {cur_lr:.2e} | {elapsed:.1f} min | ckpt saved")

    # ── Best checkpoint ────────────────────────────────────────────
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        path = save_checkpoint(model, optimizer, scheduler,
                               epoch, epoch_loss, tag="best")
        wandb.save(path)

total_time = (time.time() - training_start) / 60
print(f"Training done in {total_time:.1f} min.  Best loss: {best_loss:.6f}")

# ── Dual-panel loss curve: linear left, log right ─────────────────────
epochs_x = list(range(1, len(train_losses) + 1))

fig_dual = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Loss — Linear Scale", "Loss — Log Scale")
)
for col, log_y in [(1, False), (2, True)]:
    fig_dual.add_trace(go.Scatter(
        x=epochs_x, y=train_losses, mode="lines",
        name="Train Loss",
        line=dict(color="#636EFA" if col == 1 else "#FF7F0E", width=2),
        showlegend=(col == 1)
    ), row=1, col=col)
    if log_y:
        fig_dual.update_yaxes(type="log",
                              title_text="Chamfer Distance (log scale)",
                              row=1, col=col)
    else:
        fig_dual.update_yaxes(title_text="Chamfer Distance", row=1, col=col)
    fig_dual.update_xaxes(title_text="Epoch", row=1, col=col)

fig_dual.update_layout(
    title    = "PointNet++ — Training Loss (Linear & Log Scale)",
    template = "plotly_dark",
    height   = 450
)
fig_dual.show()
wandb.log({"loss_curve_dual_scale": wandb.Plotly(fig_dual)})

# ── Standalone log-scale curve ─────────────────────────────────────────
fig_log = go.Figure(go.Scatter(
    x=epochs_x, y=train_losses, mode="lines",
    line=dict(color="#FF7F0E", width=2), name="Train Loss"
))
fig_log.update_layout(
    title       = "PointNet++ — Training Loss (Log Scale)",
    xaxis_title = "Epoch",
    yaxis       = dict(type="log", title="Chamfer Distance (log scale)"),
    template    = "plotly_dark"
)
fig_log.show()
wandb.log({"loss_curve_log_scale": wandb.Plotly(fig_log)})

wandb.finish()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ghosalsohom2003 (ghosalsohom2003-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch   50 | Loss 0.011658 | LR 9.54e-04 | 11.8 min | ckpt saved
Epoch  100 | Loss 0.009804 | LR 7.81e-04 | 23.5 min | ckpt saved
Epoch  150 | Loss 0.008813 | LR 5.27e-04 | 35.0 min | ckpt saved
Epoch  200 | Loss 0.008096 | LR 2.66e-04 | 46.6 min | ckpt saved
Epoch  250 | Loss 0.007688 | LR 7.16e-05 | 58.0 min | ckpt saved
Epoch  300 | Loss 0.007637 | LR 0.00e+00 | 69.4 min | ckpt saved
Training done in 69.4 min.  Best loss: 0.007427


elapsed_min,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇██
epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇█████
lr,██████▇▇▇▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁
train_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_log,█▆▆▅▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
elapsed_min,69.36985
epoch,300
lr,0
train_loss,0.00764
train_loss_log,-2.11708


In [ ]:
# ─────────────────────────────────────────────
# PART 7: Visualize -- TRAIN SET
# Added: log-scale CD histogram alongside linear
# ─────────────────────────────────────────────
wandb.init(project="maize-pointnetpp", name="pnetpp-viz-train", resume="allow")

model = PointNetPP_Autoencoder(num_points=NUM_POINTS,
                               latent_dim=CONFIG["latent_dim"]).to(device)
load_checkpoint(os.path.join(CKPT_DIR, "pnetpp_best.pt"), model)
model.eval()

train_orig_list, train_recon_list = [], []
with torch.no_grad():
    for batch, _ in train_loader:
        batch = batch.to(device)
        recon, _ = model(batch)
        train_orig_list.append(batch.cpu().numpy())
        train_recon_list.append(recon.cpu().numpy())

train_orig  = np.concatenate(train_orig_list,  axis=0)
train_recon = np.concatenate(train_recon_list, axis=0)

train_cd = np.array([
    chamfer_distance(
        torch.tensor(train_orig[i]).unsqueeze(0),
        torch.tensor(train_recon[i]).unsqueeze(0)
    ).item()
    for i in range(len(train_orig))
])
print(f"Train -- Mean CD: {train_cd.mean():.6f} | Std: {train_cd.std():.6f}")

# Best 4 + Worst 4 3-D reconstructions
show_idx = list(np.argsort(train_cd)[:4]) + list(np.argsort(train_cd)[-4:])
labels   = [f"Best {i+1}" for i in range(4)] + [f"Worst {i+1}" for i in range(4)]

fig_train = make_subplots(
    rows=8, cols=2,
    specs=[[{"type":"scatter3d"}, {"type":"scatter3d"}]] * 8,
    subplot_titles=[
        t for lbl, idx in zip(labels, show_idx)
        for t in (f"{lbl} -- Original  (CD={train_cd[idx]:.4f})",
                  f"{lbl} -- Reconstructed")
    ],
    vertical_spacing=0.02
)
for row, idx in enumerate(show_idx, start=1):
    o, r = train_orig[idx], train_recon[idx]
    fig_train.add_trace(go.Scatter3d(
        x=o[:,0], y=o[:,1], z=o[:,2], mode="markers",
        marker=dict(size=1.5, color=o[:,2], colorscale="Viridis", showscale=False),
        showlegend=False
    ), row=row, col=1)
    fig_train.add_trace(go.Scatter3d(
        x=r[:,0], y=r[:,1], z=r[:,2], mode="markers",
        marker=dict(size=1.5, color=r[:,2], colorscale="Plasma", showscale=False),
        showlegend=False
    ), row=row, col=2)

fig_train.update_layout(
    title="PointNet++ Train Set -- Best 4 & Worst 4",
    template="plotly_dark", height=3600, margin=dict(l=10, r=10, t=60, b=10))
fig_train.show()
wandb.log({"train_reconstructions": wandb.Plotly(fig_train)})

# Dual histogram: linear + log10
fig_hist = make_subplots(rows=1, cols=2,
    subplot_titles=("CD Distribution (Linear)", "CD Distribution (Log10)"))
fig_hist.add_trace(go.Histogram(
    x=train_cd, nbinsx=40, name="Train CD", marker_color="#636EFA"), row=1, col=1)
fig_hist.add_trace(go.Histogram(
    x=np.log10(train_cd + 1e-10), nbinsx=40,
    name="Train CD (log10)", marker_color="#AB63FA"), row=1, col=2)
fig_hist.update_xaxes(title_text="Chamfer Distance", row=1, col=1)
fig_hist.update_xaxes(title_text="log10(CD)", row=1, col=2)
fig_hist.update_yaxes(title_text="Count", row=1, col=1)
fig_hist.update_layout(
    title="Train -- Chamfer Distance Distribution (Linear & Log)",
    template="plotly_dark", showlegend=False)
fig_hist.show()
wandb.log({"train_cd_hist_dual": wandb.Plotly(fig_hist)})

wandb.finish()


  Loaded -- epoch 276 | loss 0.007427
Train -- Mean CD: 0.006443 | Std: 0.001565


In [ ]:
# ─────────────────────────────────────────────
# PART 8: Visualize -- TEST SET + Comparison
# Added: log-scale box plot, log histogram
# ─────────────────────────────────────────────
wandb.init(project="maize-pointnetpp", name="pnetpp-viz-test", resume="allow")

model = PointNetPP_Autoencoder(num_points=NUM_POINTS,
                               latent_dim=CONFIG["latent_dim"]).to(device)
load_checkpoint(os.path.join(CKPT_DIR, "pnetpp_best.pt"), model)
model.eval()

test_orig_list, test_recon_list = [], []
with torch.no_grad():
    for batch, _ in test_loader:
        batch = batch.to(device)
        recon, _ = model(batch)
        test_orig_list.append(batch.cpu().numpy())
        test_recon_list.append(recon.cpu().numpy())

test_orig  = np.concatenate(test_orig_list,  axis=0)
test_recon = np.concatenate(test_recon_list, axis=0)

test_cd = np.array([
    chamfer_distance(
        torch.tensor(test_orig[i]).unsqueeze(0),
        torch.tensor(test_recon[i]).unsqueeze(0)
    ).item()
    for i in range(len(test_orig))
])
print(f"Test  -- Mean CD: {test_cd.mean():.6f} | Std: {test_cd.std():.6f}")

# Best 4 + Worst 4 3-D reconstructions
show_idx = list(np.argsort(test_cd)[:4]) + list(np.argsort(test_cd)[-4:])
labels   = [f"Best {i+1}" for i in range(4)] + [f"Worst {i+1}" for i in range(4)]

fig_test = make_subplots(
    rows=8, cols=2,
    specs=[[{"type":"scatter3d"}, {"type":"scatter3d"}]] * 8,
    subplot_titles=[
        t for lbl, idx in zip(labels, show_idx)
        for t in (f"{lbl} -- Original  (CD={test_cd[idx]:.4f})",
                  f"{lbl} -- Reconstructed")
    ],
    vertical_spacing=0.02
)
for row, idx in enumerate(show_idx, start=1):
    o, r = test_orig[idx], test_recon[idx]
    fig_test.add_trace(go.Scatter3d(
        x=o[:,0], y=o[:,1], z=o[:,2], mode="markers",
        marker=dict(size=1.5, color=o[:,2], colorscale="Viridis", showscale=False),
        showlegend=False
    ), row=row, col=1)
    fig_test.add_trace(go.Scatter3d(
        x=r[:,0], y=r[:,1], z=r[:,2], mode="markers",
        marker=dict(size=1.5, color=r[:,2], colorscale="Plasma", showscale=False),
        showlegend=False
    ), row=row, col=2)

fig_test.update_layout(
    title="PointNet++ Test Set -- Best 4 & Worst 4",
    template="plotly_dark", height=3600, margin=dict(l=10, r=10, t=60, b=10))
fig_test.show()
wandb.log({"test_reconstructions": wandb.Plotly(fig_test)})

# Dual histogram
fig_hist_test = make_subplots(rows=1, cols=2,
    subplot_titles=("Test CD (Linear)", "Test CD (Log10)"))
fig_hist_test.add_trace(go.Histogram(
    x=test_cd, nbinsx=40, name="Test CD", marker_color="#EF553B"), row=1, col=1)
fig_hist_test.add_trace(go.Histogram(
    x=np.log10(test_cd + 1e-10), nbinsx=40,
    name="Test CD (log10)", marker_color="#FFA15A"), row=1, col=2)
fig_hist_test.update_xaxes(title_text="Chamfer Distance", row=1, col=1)
fig_hist_test.update_xaxes(title_text="log10(CD)", row=1, col=2)
fig_hist_test.update_layout(
    title="Test -- Chamfer Distance Distribution (Linear & Log)",
    template="plotly_dark", showlegend=False)
fig_hist_test.show()
wandb.log({"test_cd_hist_dual": wandb.Plotly(fig_hist_test)})

# Train vs Test box plot -- LOG SCALE y-axis
fig_box = go.Figure()
fig_box.add_trace(go.Box(y=train_cd, name="Train (seen)",
                         marker_color="#636EFA", boxmean=True))
fig_box.add_trace(go.Box(y=test_cd,  name="Test (unseen)",
                         marker_color="#EF553B", boxmean=True))
fig_box.update_layout(
    title       = "PointNet++ -- Train vs Test CD (Log Scale)",
    yaxis       = dict(type="log", title="Chamfer Distance (log scale)"),
    template    = "plotly_dark"
)
fig_box.show()
wandb.log({"train_vs_test_cd_log": wandb.Plotly(fig_box)})

# Latent space PCA
train_latents, test_latents = [], []
with torch.no_grad():
    for batch, _ in train_loader:
        _, z = model(batch.to(device))
        train_latents.append(z.cpu().numpy())
    for batch, _ in test_loader:
        _, z = model(batch.to(device))
        test_latents.append(z.cpu().numpy())

train_latents = np.concatenate(train_latents, axis=0)
test_latents  = np.concatenate(test_latents,  axis=0)
latents   = np.concatenate([train_latents, test_latents], axis=0)
split_lbl = (["Train"] * len(train_latents) + ["Test"] * len(test_latents))

print(f"Latents shape : {latents.shape}")
assert latents.shape[0] == len(split_lbl), "Length mismatch!"

latent_2d = PCA(n_components=2).fit_transform(latents)
latent_3d = PCA(n_components=3).fit_transform(latents)

fig_pca2 = px.scatter(
    x=latent_2d[:,0], y=latent_2d[:,1], color=split_lbl,
    color_discrete_map={"Train":"#636EFA","Test":"#EF553B"},
    title="Latent Space PCA 2D -- Train vs Test",
    labels={"x":"PC1","y":"PC2","color":"Split"}, template="plotly_dark")
fig_pca2.show()
wandb.log({"latent_pca_2d": wandb.Plotly(fig_pca2)})

fig_pca3 = px.scatter_3d(
    x=latent_3d[:,0], y=latent_3d[:,1], z=latent_3d[:,2], color=split_lbl,
    color_discrete_map={"Train":"#636EFA","Test":"#EF553B"},
    title="Latent Space PCA 3D -- Train vs Test",
    labels={"x":"PC1","y":"PC2","z":"PC3","color":"Split"}, template="plotly_dark")
fig_pca3.update_traces(marker=dict(size=3))
fig_pca3.show()
wandb.log({"latent_pca_3d": wandb.Plotly(fig_pca3)})

# Summary table
wandb.log({
    "evaluation_summary": wandb.Table(
        columns=["Split","Samples","Mean CD","Std CD","Min CD","Max CD"],
        data=[
            ["Train", len(train_cd),
             round(float(train_cd.mean()),6), round(float(train_cd.std()),6),
             round(float(train_cd.min()),6),  round(float(train_cd.max()),6)],
            ["Test",  len(test_cd),
             round(float(test_cd.mean()),6),  round(float(test_cd.std()),6),
             round(float(test_cd.min()),6),   round(float(test_cd.max()),6)],
        ]
    )
})

wandb.finish()
print("All done.")


  Loaded -- epoch 276 | loss 0.007427
Test  -- Mean CD: 0.008613 | Std: 0.006135


Latents shape : (1041, 512)


All done.
